# DL Trading with MLOps - Example Usage

This notebook demonstrates the usage of the DL Trading system.

## 1. Setup and Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import TechnicalIndicators, MultiTimeframeFeatures
from src.models import CNNModel, ModelTrainer
from src.monitoring import DriftDetector
from src.backtesting import BacktestEngine, TradingStrategy

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 2. Load and Explore Data

In [ ]:
# Load raw data
data = pd.read_csv('../data/raw_data.csv', index_col=0, parse_dates=True)
print(f"Data shape: {data.shape}")
print(f"Date range: {data.index[0]} to {data.index[-1]}")
data.head()

In [ ]:
# Plot price data
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['Close'], label='Close Price')
plt.title('Price History')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

## 3. Feature Engineering

In [ ]:
# Add technical indicators
indicators = TechnicalIndicators(data)
data_with_indicators = indicators.add_all_indicators()

print(f"Features added: {len(data_with_indicators.columns) - len(data.columns)}")
print(f"Total features: {len(data_with_indicators.columns)}")

In [ ]:
# Plot some indicators
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price with moving averages
axes[0].plot(data_with_indicators.index, data_with_indicators['Close'], label='Close')
axes[0].plot(data_with_indicators.index, data_with_indicators['SMA_20'], label='SMA 20')
axes[0].plot(data_with_indicators.index, data_with_indicators['SMA_50'], label='SMA 50')
axes[0].set_title('Price with Moving Averages')
axes[0].legend()

# RSI
axes[1].plot(data_with_indicators.index, data_with_indicators['RSI_14'], label='RSI')
axes[1].axhline(y=70, color='r', linestyle='--', label='Overbought')
axes[1].axhline(y=30, color='g', linestyle='--', label='Oversold')
axes[1].set_title('RSI Indicator')
axes[1].legend()

# MACD
axes[2].plot(data_with_indicators.index, data_with_indicators['MACD'], label='MACD')
axes[2].plot(data_with_indicators.index, data_with_indicators['MACD_Signal'], label='Signal')
axes[2].set_title('MACD')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Load Trained Model and Make Predictions

In [ ]:
# Load model (if available)
from tensorflow import keras
import joblib

try:
    model = keras.models.load_model('../models/best_cnn_model.keras')
    scaler = joblib.load('../models/best_cnn_model_scaler.pkl')
    print("Model loaded successfully!")
    print(f"Model input shape: {model.input_shape}")
except:
    print("Model not found. Please train the model first using scripts/03_train_model.py")

## 5. Data Drift Analysis

In [ ]:
# Load engineered features
features = pd.read_csv('../data/engineered_features.csv', index_col=0, parse_dates=True)

# Split into reference and current
split_point = int(len(features) * 0.8)
reference_data = features[:split_point].dropna()
current_data = features[split_point:].dropna()

print(f"Reference data: {len(reference_data)} rows")
print(f"Current data: {len(current_data)} rows")

In [ ]:
# Initialize drift detector
detector = DriftDetector(reference_data=reference_data)

# Detect drift
drift_results = detector.detect_drift_all_features(current_data)

print("Drift Detection Summary:")
print(f"Total features: {drift_results['summary']['total_features']}")
print(f"Drifted features: {drift_results['summary']['drifted_features']}")
print(f"Drift percentage: {drift_results['summary']['drift_percentage']:.2f}%")

## 6. Backtest Results Analysis

In [ ]:
# Load backtest results
try:
    trades = pd.read_csv('../reports/backtest_trades.csv')
    portfolio = pd.read_csv('../reports/portfolio_history.csv', parse_dates=['timestamp'])
    
    print(f"Total trades: {len(trades)}")
    print(f"\nTrade summary:")
    print(trades.groupby('type').size())
except:
    print("Backtest results not found. Run scripts/06_run_backtest.py first.")

In [ ]:
# Plot portfolio value
if 'portfolio' in locals():
    plt.figure(figsize=(14, 6))
    plt.plot(portfolio['timestamp'], portfolio['portfolio_value'])
    plt.title('Portfolio Value Over Time')
    plt.xlabel('Date')
    plt.ylabel('Portfolio Value ($)')
    plt.grid(True)
    plt.show()